In [14]:
## Step 1: Load Data and Check Row Count
import pandas as pd, sqlite3
df = pd.read_csv("SQL_Sales_Dataset_200_Rows.xlsx - Sheet1.csv")
conn = sqlite3.connect(":memory:")
df.to_sql("sales", conn, index=False)

def q(sql):
    return pd.read_sql(sql, conn)

q("SELECT COUNT(*) AS total_orders FROM sales")

,total_orders
0,200


In [15]:
## Query 1: Top 10 Customers by Total Spending
q("""
SELECT customer_name, COUNT(order_id) AS total_orders, SUM(total_price) AS total_spent
FROM sales
GROUP BY customer_name
ORDER BY total_spent DESC
LIMIT 10
""")

,customer_name,total_orders,total_spent
0,Lynn Garrison,1,47940
1,Debbie Turner,1,44990
2,Megan Charles,1,44620
3,Michelle Beltran,1,42471
4,Rick Sanford,1,41211
5,Patrick Watson,1,37680
6,Christopher Meyer,1,37320
7,Todd Barnes,1,36152
8,Stephanie Martinez,1,34576
9,Kimberly Alvarado,1,31752


In [16]:
## Query 2: Overall Summary and Average Order Value
q("""
SELECT COUNT(*) AS total_orders,
       SUM(total_price) AS total_revenue,
       ROUND(AVG(total_price), 2) AS avg_order_value
FROM sales
""")

,total_orders,total_revenue,avg_order_value
0,200,2420107,12100.53


In [17]:
## Query 3:Average Order Value per Customer
q("""
SELECT customer_name,
       ROUND(AVG(total_price), 2) AS avg_order_value
FROM sales
GROUP BY customer_name
ORDER BY avg_order_value DESC
LIMIT 10
""")

,customer_name,avg_order_value
0,Lynn Garrison,47940.0
1,Debbie Turner,44990.0
2,Megan Charles,44620.0
3,Michelle Beltran,42471.0
4,Rick Sanford,41211.0
5,Patrick Watson,37680.0
6,Christopher Meyer,37320.0
7,Todd Barnes,36152.0
8,Stephanie Martinez,34576.0
9,Kimberly Alvarado,31752.0


In [18]:
## Query 4:Average Order Value by Category
q("""
SELECT category,
       COUNT(*) AS orders,
       SUM(total_price) AS revenue,
       ROUND(AVG(total_price), 2) AS avg_order_value
FROM sales
GROUP BY category
ORDER BY revenue DESC
""")

,category,orders,revenue,avg_order_value
0,Furniture,59,714399,12108.46
1,Grocery,49,672147,13717.29
2,Electronics,52,549302,10563.50
3,Clothing,40,484259,12106.48


In [19]:
## Query 5:Revenue and Average Order Value by Region
q("""
SELECT region,
       SUM(total_price) AS revenue,
       ROUND(AVG(total_price), 2) AS avg_order_value
FROM sales
GROUP BY region
ORDER BY revenue DESC
""")

,region,revenue,avg_order_value
0,South,808651,12252.29
1,East,619250,12142.16
2,West,599367,13938.77
3,North,392839,9820.98


In [20]:
## Query 6:Subquery, Orders Above the Average
q("""
SELECT customer_name, total_price
FROM sales
WHERE total_price > (SELECT AVG(total_price) FROM sales)
ORDER BY total_price DESC
""")

,customer_name,total_price
0,Lynn Garrison,47940
1,Debbie Turner,44990
2,Megan Charles,44620
3,Michelle Beltran,42471
4,Rick Sanford,41211
...,...,...
76,Roy Stewart,13300
77,Melissa Ruiz,13300
78,Sheila Booker,12817
79,Laura Smith,12222


In [21]:
## Query 7:CASE Statement, Order Value Segments
q("""
SELECT order_id, customer_name, total_price,
       CASE
           WHEN total_price >= 20000 THEN 'High Value'
           WHEN total_price >= 5000  THEN 'Medium Value'
           ELSE 'Low Value'
       END AS order_segment
FROM sales
ORDER BY total_price DESC
""")

,order_id,customer_name,total_price,order_segment
0,1125,Lynn Garrison,47940,High Value
1,1019,Debbie Turner,44990,High Value
2,1168,Megan Charles,44620,High Value
3,1160,Michelle Beltran,42471,High Value
4,1120,Rick Sanford,41211,High Value
...,...,...,...,...
195,1060,Rebecca Lozano,390,Low Value
196,1023,Nicholas Ruiz,262,Low Value
197,1038,Walter Pearson,253,Low Value
198,1108,Joshua Myers,243,Low Value


In [22]:
## Query 8:Orders per Segment
q("""
SELECT CASE
           WHEN total_price >= 20000 THEN 'High Value'
           WHEN total_price >= 5000  THEN 'Medium Value'
           ELSE 'Low Value'
       END AS order_segment,
       COUNT(*) AS orders
FROM sales
GROUP BY order_segment
""")

,order_segment,orders
0,High Value,48
1,Low Value,74
2,Medium Value,78


In [23]:
## Query 9:Monthly Revenue Trend
q("""
SELECT SUBSTR(order_date, 1, 7) AS month,
       SUM(total_price) AS revenue
FROM sales
GROUP BY SUBSTR(order_date, 1, 7)
ORDER BY month
""")

,month,revenue
0,2024-12,62504
1,2025-01,424143
2,2025-02,327627
3,2025-03,567482
4,2025-04,324322
5,2025-05,410061
6,2025-06,303968
